![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 22 -- Lab 1: Convolution by Hand

The deck slid a 3 x 3 kernel over a 5 x 5 patch of a handwritten 3 and found 772 where the stroke began. You rebuild that in NumPy: one window, then two loops, then padding and stride, then the output-shape formula, each checked against PyTorch. Then you run hand-made kernels over a real photograph, watch the feature map move with the photo, pool it, and count the dials a convolution needs against the dials a fully connected layer needs.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Two small things that need no download: the 5 x 5 patch of the deck (MNIST image 7, rows 10 to 14 and columns 10 to 14, typed into the setup cell) and scikit-learn's sample photograph `china.jpg` (427 x 640 pixels, three channels), loaded with `load_sample_image`.

---
# Setup

In [ ]:
# numpy, matplotlib, scikit-learn and torch are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q numpy matplotlib scikit-learn torch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_sample_image
np.set_printoptions(linewidth=120)

In [ ]:
patch = np.array([[  0,   0,   0,   0,   0],
                  [  0,   0,   0,   0,   0],
                  [  0,   0,   0,   0,  91],
                  [125, 193, 193, 193, 253],
                  [252, 252, 252, 252, 253]], dtype=float)   # the deck's 5 x 5 patch of the digit 3
K_h = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=float)    # horizontal-edge kernel
K_v = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)    # vertical-edge kernel
plt.imshow(patch, cmap="gray_r", vmin=0, vmax=255); plt.title("the patch: white above, ink below"); plt.axis("off"); plt.show()

---
# Part 1 -- One window, then every window

The *Definition: A Kernel Slides Over the Image* and *The Math, With Numbers* slides.

## Task 1: One window by hand

Take the centre window `win = patch[1:4, 1:4]` (rows 1 to 3, columns 1 to 3). Compute the sum of products with the horizontal kernel, `np.sum(win * K_h)`, as `y_h`, and with the vertical kernel as `y_v`. Print both: the deck says 772 and 0.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert y_h == 772 and y_v == 0, 'one window: multiply the 3 x 3 window by the kernel element by element, then sum the nine products'
print('Task 1 passed')

## Task 2: Two loops: the convolution

Write `conv2d(X, K)` for a 2-D array `X` and a kernel `K` (no padding, stride 1): the output has `X.shape[0] - K.shape[0] + 1` rows and `X.shape[1] - K.shape[1] + 1` columns; for every output position `(i, j)` take the window `X[i:i + kh, j:j + kw]` and store the sum of products. Run it on `patch` with `K_h` and print the 3 x 3 result: the deck's `[[0, 0, 91], [704, 772, 832], [1008, 1008, 918]]`.

Syntax hint (the shape of the call, not the answer):

```python
def conv2d(X, K):
    out = np.zeros((rows_out, cols_out))
    for i in range(rows_out):
        for j in range(cols_out):
            out[i, j] = ...   # the window at (i, j) times K, summed
    return out
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert fmap_h.shape == (3, 3) and np.allclose(fmap_h, [[0, 0, 91], [704, 772, 832], [1008, 1008, 918]]), 'conv2d: window X[i:i+kh, j:j+kw], multiplied by K, summed, for every (i, j)'
print('Task 2 passed')

## Task 3: Check against PyTorch

PyTorch wants four axes, (batch, channels, height, width). Build `x = torch.tensor(patch)[None, None].float()` and `k = torch.tensor(K_h)[None, None].float()`, run `F.conv2d(x, k)`, and compare its `[0, 0]` slice (as a NumPy array) with your `fmap_h` using `np.allclose`. Print the result of the comparison and the shape of the PyTorch output.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert np.allclose(y_torch[0, 0].numpy(), fmap_h), 'PyTorch: F.conv2d(x, k) with x of shape (1, 1, 5, 5) and k of shape (1, 1, 3, 3)'
print('Task 3 passed')

---
# Part 2 -- Padding, stride and the output shape

The *Padding*, *Stride* and *Output-Shape Formula* slides.

## Task 4: Padding and stride

Write `conv2d_ps(X, K, padding, stride)`: pad `X` with `np.pad(X, padding)` (zeros around the border), compute the output size as `(padded_rows - kh) // stride + 1` (and the same for columns), and take the window at `Xp[i * stride:i * stride + kh, j * stride:j * stride + kw]`. Test it on the practice task of the deck: `X = np.arange(1, 17).reshape(4, 4)`, `K = np.array([[1, 0], [-1, 1]])`, padding 1, stride 2. Expected `[[1, 1, -4], [9, 7, -4], [0, 14, 16]]`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert np.allclose(out_ps, [[1, 1, -4], [9, 7, -4], [0, 14, 16]]), 'conv2d_ps: np.pad first, then windows every `stride` rows and columns of the padded array'
assert np.allclose(conv2d_ps(patch, K_h, 1, 1)[1:4, 1:4], fmap_h), 'conv2d_ps with padding 1, stride 1 must contain the unpadded result in its middle'
print('Task 4 passed')

## Task 5: The output-shape formula

Write `out_size(n, k, p, s)` returning `(n + 2 * p - k) // s + 1`. Print it for the deck's table: `(28, 3, 0, 1)`, `(28, 3, 1, 1)`, `(28, 5, 2, 1)`, `(28, 3, 1, 2)`, `(28, 5, 0, 2)`; expected 26, 28, 28, 14, 12. Then confirm one row with PyTorch: `nn.Conv2d(1, 8, kernel_size=3, padding=1, stride=2)(torch.zeros(1, 1, 28, 28)).shape`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert [out_size(*r) for r in [(28, 3, 0, 1), (28, 3, 1, 1), (28, 5, 2, 1), (28, 3, 1, 2), (28, 5, 0, 2)]] == [26, 28, 28, 14, 12], 'out_size: floor((n + 2p - k) / s) + 1'
print('Task 5 passed')

---
# Part 3 -- Kernels on a real photo

The *Hand-Made Kernels on a Real Photo* and *Why Sharing Wins* slides. The photo is big, so use `F.conv2d` (your loops would take a while).

In [ ]:
photo = load_sample_image("china.jpg")                 # (427, 640, 3), values 0..255
grey = photo.mean(axis=2) / 255.0                        # one channel, values 0..1
g = torch.tensor(grey, dtype=torch.float32)[None, None]  # (1, 1, 427, 640)
print(photo.shape, grey.shape, g.shape)

## Task 6: Edge maps of the photo

Write `apply_kernel(img4d, K)` that returns `F.conv2d(img4d, torch.tensor(K, dtype=torch.float32)[None, None], padding=1)[0, 0].numpy()`. Apply it to `g` with `K_v` and with `K_h`, and show three panels side by side: the grey photo, `np.abs(edges_v)` and `np.abs(edges_h)` (all with `cmap="gray"`, axes off, titles). Which kernel finds the pillars, which the roof lines?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert edges_v.shape == (427, 640) and edges_h.shape == (427, 640), 'apply_kernel: padding=1 keeps the 427 x 640 size; take [0, 0] to drop the batch and channel axes'
print('Task 6 passed')

## Task 7: The feature map moves with the photo

Shift the photo 20 pixels to the right with `np.roll(grey, 20, axis=1)`, turn it into a 4-D tensor like `g`, and apply `K_v` to it as `edges_shifted`. Then check that the shifted photo's edge map is the original edge map shifted by 20 pixels: `np.allclose(np.roll(edges_v, 20, axis=1)[:, 40:-2], edges_shifted[:, 40:-2], atol=1e-4)` (the first 40 and the last 2 columns are skipped: the roll wraps the right border around, and the zero padding sits at different pixels there). Print the result. This is the *shared weights* idea: the same kernel finds the same pattern wherever it moved to.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert same, 'shift test: convolve the rolled photo, then compare with the rolled feature map on columns 40 to -2'
print('Task 7 passed')

---
# Part 4 -- Pooling and the dials

The *Definition: Max Pooling* and *Counting the Dials* slides.

## Task 8: Max pooling by hand

Write `maxpool2(X)` that takes a 2-D array with even sides and returns an array half as tall and half as wide, each entry the largest number of its 2 x 2 block (two loops over the output, or `X.reshape(h // 2, 2, w // 2, 2).max(axis=(1, 3))`). Test it on the deck's block `B = np.array([[1, 3, 2, 0], [4, 6, 1, 1], [0, 2, 5, 3], [1, 1, 2, 9]])`: expected `[[6, 2], [2, 9]]`. Compare with `nn.MaxPool2d(2)` on `torch.tensor(B)[None, None].float()`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert np.array_equal(pooled, [[6, 2], [2, 9]]), 'maxpool2: the maximum of every non-overlapping 2 x 2 block'
print('Task 8 passed')

## Task 9: Pool the edge map twice

Apply `maxpool2` to `e1 = np.abs(edges_v)[:426, :]` (one row dropped to make the height even) as `e2`, then once more to `e2[:212, :]` (again one odd row dropped) as `e3`. Print the three shapes and show the three maps side by side. Do the pillars survive two poolings?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert e2.shape == (213, 320) and e3.shape == (106, 160), 'pooling halves each side; drop the odd row before the second pooling (213 -> 106)'
print('Task 9 passed')

## Task 10: Count the dials

Count the parameters of `nn.Conv2d(3, 16, 3)` and of `nn.Linear(3072, 16)` with `sum(p.numel() for p in layer.parameters())`, as `n_conv` and `n_lin`. Then do the same for `nn.Conv2d(3, 16, 3)` applied to a 224 x 224 photo (same layer, so the same number) and `nn.Linear(3 * 224 * 224, 16)`. Print all four with the deck's formula next to them: `16 * (3 * 9) + 16` and `3072 * 16 + 16`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert n_conv == 448 and n_lin == 49168, 'dials: C_out * (C_in * k * k) + C_out for the convolution, inputs * outputs + outputs for Linear'
print('Task 10 passed')

---
## Task 11: Reflect

Your `conv2d` agreed with `F.conv2d` to the last digit. In one sentence, what does a convolutional layer learn that your version has fixed? The shift test passed: say in your own words why the feature map moved with the photo, and name the slide idea it illustrates. The convolution needed 448 dials for a 32 x 32 photo and still 448 for a 224 x 224 one. What grew instead, and why does that make a GPU useful?

*Your answers here*

---
## Conclusion

- **Convolution**: you computed one window by hand (772 and 0), then every window with two loops, and PyTorch's `F.conv2d` agreed exactly.
- **Padding, stride and shape**: `np.pad` and a stride in the loops reproduced the practice task, and `out_size` reproduced the deck's table.
- **Kernels as detectors**: the Sobel kernels found the pillars and the roof lines of a real photo, and the feature map shifted with the photo because the same dials are used everywhere.
- **Pooling and dials**: max pooling kept the edges while halving the map twice, and a convolution's 448 dials did not grow with the image while the fully connected layer's did.

---

Made By **Sattam Altwaim**